# Lesson 2 Build a semantic search engine

## Start ollama by docker compose

In [1]:
!docker compose up -d ollama

 Container ollama  Running


## Pull Meta-Llama-3.1-8B-Claude-GGUF model from Hugging Face

In [2]:
!docker compose exec ollama ollama pull hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M

pulling manifest 
pulling e5143516efe0: 100% ▕██████████████████▏ 4.9 GB                         
pulling 783adfd1d253: 100% ▕██████████████████▏  976 B                         
pulling 1a9f0f5ed111: 100% ▕██████████████████▏   22 B                         
pulling d9b87732a16b: 100% ▕██████████████████▏  552 B                         
verifying sha256 digest 
writing manifest 
success 


## Set Ollama environment variables

In [3]:
import os

os.environ["OLLAMA_API_BASE"] = "http://localhost:11434"
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_TRACING"] = "true"

## Load PDF

LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has three attributes:

page_content: a string representing the content;
metadata: a dict containing arbitrary metadata;
id: (optional) a string identifier for the document.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

file_paths = ["../docs/LLM-Based Conversational Assistants that Help Users.pdf", "../docs/HDFLOW ENHANCING LLM COMPLEX PROBLEMSOLVING.pdf"]

docs = [PyPDFLoader(file_path).load() for file_path in file_paths]
print(len(docs))

print(docs)

2
[[Document(metadata={'producer': 'pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creator': 'LaTeX with acmart 2023/10/14 v1.92 Typesetting articles for the Association for Computing Machinery and hyperref 2023-04-22 v7.00x Hypertext links for LaTeX', 'creationdate': '2024-12-23T01:13:47+00:00', 'moddate': '2024-12-23T01:13:47+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': 'Thinking Assistants: LLM-Based Conversational Assistants that Help Users Think By Asking rather than Answering', 'trapped': '/False', 'source': '../docs/LLM-Based Conversational Assistants that Help Users.pdf', 'total_pages': 29, 'page': 0, 'page_label': '1'}, page_content='Thinking Assistants: LLM-Based Conversational Assistants that Help Users\nThink By Asking rather than Answering\nSOYA PARK,Emory University, USA\nHARI SUBRAMONYAM, Stanford University, USA\nCHINMAY KULKARNI, Emory Univ

## Split content

For both information retrieval and downstream question-answering purposes, a page may be too coarse a representation. Our goal in the end will be to retrieve Document objects that answer an input query, and further splitting our PDF will help ensure that the meanings of relevant portions of the document are not "washed out" by surrounding text.

We can use text splitters for this purpose. Here we will use a simple text splitter that partitions based on characters. We will split our documents into chunks of 1000 characters with 200 characters of overlap between chunks. The overlap helps mitigate the possibility of separating a statement from important context related to it. We use the RecursiveCharacterTextSplitter, which will recursively split the document using common separators like new lines until each chunk is the appropriate size. This is the recommended text splitter for generic text use cases.

We set add_start_index=True so that the character index where each split Document starts within the initial Document is preserved as metadata attribute “start_index”.

See [this guide](https://python.langchain.com/docs/how_to/document_loader_pdf/) for more detail about working with PDFs, including how to extract text from specific sections and images.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)

all_splits = [text_splitter.split_documents(doc) for doc in docs]
print(len(all_splits))

flattened_all_splits = [item for sublist in all_splits for item in sublist]

print(len(flattened_all_splits))
print(flattened_all_splits[0])



2
266
page_content='Thinking Assistants: LLM-Based Conversational Assistants that Help Users
Think By Asking rather than Answering
SOYA PARK,Emory University, USA
HARI SUBRAMONYAM, Stanford University, USA
CHINMAY KULKARNI, Emory University, USA
Many AI systems focus solely on providing solutions or explaining outcomes. However, complex tasks like research and strategic
thinking often benefit from a more comprehensive approach to augmenting the thinking process rather than passively getting
information. We introduce the concept of “Thinking Assistants”, a new genre of assistants that help users improve decision-making
with a combination of asking reflection questions based on expert knowledge. Through our lab study (N=80), these Large Language
Model (LLM) based Thinking Assistants were better able to guide users to make important decisions, compared with conversational
agents that only asked questions, provided advice, or neither.' metadata={'producer': 'pdfTeX, Version 3.141592653-2.6

## Load embedding model

In [6]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="hf.co/nomic-ai/nomic-embed-text-v1.5-GGUF:F16")

## Bind embeddings and add documenets in vector store

LangChain VectorStore objects contain methods for adding text and Document objects to the store, and querying them using various similarity metrics. They are often initialized with embedding models, which determine how text data is translated to numeric vectors.

LangChain includes a suite of integrations with different vector store technologies. Some vector stores are hosted by a provider (e.g., various cloud providers) and require specific credentials to use; some (such as Postgres) run in separate infrastructure that can be run locally or via a third-party; others can run in-memory for lightweight workloads. Let's select InMemoryVectorStore

In [7]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

Having instantiated our vector store, we can now index the documents.

In [8]:
ids = vector_store.add_documents(documents=flattened_all_splits)

print(ids)

['00b782cc-9531-409b-a4bd-23941df69cd9', 'c4d521e4-488d-46e0-8379-14864627a9a2', 'e437db52-e32e-46a8-b435-8ba9552c2261', '4b471047-6d42-4a84-8885-1cfd35aad80b', '351fd83f-f683-4530-a964-510c575bb112', 'c7ced861-5835-42f9-b6d9-ff35f64ff01c', '0feeb462-22d9-4d08-ac55-86fc0d1d2af3', 'a81cf8c8-4602-4441-8977-20f042fa9b9e', '23d2b39d-2038-48ae-9350-6ef48357d81f', '0f108c3e-4242-4d7f-aff0-7f18028f4f54', '0f3076d6-3823-4302-80c8-2f90e46fbc5d', '2b2883ef-fce2-416a-b446-733b10537d80', '54b69dcd-2008-49d8-856b-5da93cfe8986', 'cdd50bdf-c08b-4f66-b80f-c9f216ce7f62', 'eb6c47d0-ec0c-4d33-9c45-f9c175398f00', '0c44a68e-6786-4476-9fe3-a5af954c73fe', 'fae10b90-315c-4ebf-8e53-7333785bed16', 'e26bde95-2677-4990-9298-f16755627221', '4091c09a-e919-4af9-8e6a-428ab32459bb', '80c3a345-50f5-41be-91c4-dab0eb631ef0', '99caf701-bff4-49b9-b61f-6a03b408616d', '2c3541a0-f34a-4a8b-bdf5-741d1389e143', 'da98b81f-6478-40bb-8f63-48f6820996ab', 'a4e80d14-51e8-4207-b9a4-f1d25f85346d', '3fac3e92-5642-4425-8817-b1dc9967b514',

Embeddings typically represent text as a "dense" vector such that texts with similar meanings are geometrically close. This lets us retrieve relevant information just by passing in a question, without knowledge of any specific key-terms used in the document.

In [9]:
results = vector_store.similarity_search(
    "How many distribution centers does Nike have in the US?"
)

print(results[0])

page_content='problems.
Integration with Symbolic Reasoning Systems. Our dynamic workflow approach seamlessly inte-
grates specialized language models and symbolic reasoning tools, enabling LLMs to tackle complex
problems more effectively. However, there is significant potential to extend this integration to more
advanced symbolic reasoning systems, such as Lean 9 for mathematical theorem proving or other
domain-specific tools. Moreover, integrating our approach with tools such as search engines and
web browsers could enable LLMs to access and utilize external resources, further amplifying their
problem-solving abilities to broader applications. By incorporating more powerful tools into the
dynamic workflow, we can expand the range of problems that LLMs can solve.
9https://lean-lang.org/
11' metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-09-27T00:15:41+00:00', 'author': '', 'keywords': '', 'moddate': '2024-09-27T00:15:41+00:00', 'ptex.fu

In [10]:
results = await vector_store.asimilarity_search("When was Nike incorporated?")

print(results[0])

page_content='Tricky Terrain of Managing Up by Leveraging One’s Motivation to Get Things Done. ACM Transactions on Computer-Human Interaction (2024).
[44] James W Pennebaker, Martha E Francis, and Roger J Booth. 2001. Linguistic inquiry and word count: LIWC 2001. Mahway: Lawrence Erlbaum
Associates 71, 2001 (2001), 2001.
[45] John Wesley Robb. 1967. Self-Discovery and the Role of the Counselor. The Personnel and Guidance Journal 45 (1967), 1008–1011. https:
//api.semanticscholar.org/CorpusID:143714630
[46] Annabel Rothschild, Amanda Meng, Carl DiSalvo, Britney Johnson, Ben Rydal Shapiro, and Betsy DiSalvo. 2022. Interrogating Data Work as a
Community of Practice. Proc. ACM Hum.-Comput. Interact. 6, CSCW2, Article 307 (nov 2022), 28 pages. https://doi.org/10.1145/3555198
[47] Paul G Schempp and Sophie Woorons Johnson. 2006. Learning to see: Developing the perception of an expert teacher. Journal of Physical Education,
Recreation & Dance 77, 6 (2006), 29–33.' metadata={'producer': 'pdfTe

Return scores:

In [11]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("What was Nike's revenue in 2023?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.4523764785673559

page_content='Tricky Terrain of Managing Up by Leveraging One’s Motivation to Get Things Done. ACM Transactions on Computer-Human Interaction (2024).
[44] James W Pennebaker, Martha E Francis, and Roger J Booth. 2001. Linguistic inquiry and word count: LIWC 2001. Mahway: Lawrence Erlbaum
Associates 71, 2001 (2001), 2001.
[45] John Wesley Robb. 1967. Self-Discovery and the Role of the Counselor. The Personnel and Guidance Journal 45 (1967), 1008–1011. https:
//api.semanticscholar.org/CorpusID:143714630
[46] Annabel Rothschild, Amanda Meng, Carl DiSalvo, Britney Johnson, Ben Rydal Shapiro, and Betsy DiSalvo. 2022. Interrogating Data Work as a
Community of Practice. Proc. ACM Hum.-Comput. Interact. 6, CSCW2, Article 307 (nov 2022), 28 pages. https://doi.org/10.1145/3555198
[47] Paul G Schempp and Sophie Woorons Johnson. 2006. Learning to see: Developing the perception of an expert teacher. Journal of Physical Education,
Recreation & Dance 77, 6 (2006), 29–33.' m

## Retriever

LangChain VectorStore objects do not subclass Runnable. LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations). Although we can construct retrievers from vector stores, retrievers can interface with non-vector store sources of data, as well (such as external APIs).

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [12]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import chain

@chain
def retriever(query: str) -> List[Document]:
    result = vector_store.similarity_search(query, k=1)
    return result


retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)

[[Document(id='ccd58660-b8d5-42d2-ac82-5e384ffadf7a', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-09-27T00:15:41+00:00', 'author': '', 'keywords': '', 'moddate': '2024-09-27T00:15:41+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../docs/HDFLOW ENHANCING LLM COMPLEX PROBLEMSOLVING.pdf', 'total_pages': 27, 'page': 10, 'page_label': '11', 'start_index': 3244}, page_content='problems.\nIntegration with Symbolic Reasoning Systems. Our dynamic workflow approach seamlessly inte-\ngrates specialized language models and symbolic reasoning tools, enabling LLMs to tackle complex\nproblems more effectively. However, there is significant potential to extend this integration to more\nadvanced symbolic reasoning systems, such as Lean 9 for mathematical theorem proving or other\ndomain-specific tools. Moreover, integratin

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [13]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)

[[Document(id='ccd58660-b8d5-42d2-ac82-5e384ffadf7a', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-09-27T00:15:41+00:00', 'author': '', 'keywords': '', 'moddate': '2024-09-27T00:15:41+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../docs/HDFLOW ENHANCING LLM COMPLEX PROBLEMSOLVING.pdf', 'total_pages': 27, 'page': 10, 'page_label': '11', 'start_index': 3244}, page_content='problems.\nIntegration with Symbolic Reasoning Systems. Our dynamic workflow approach seamlessly inte-\ngrates specialized language models and symbolic reasoning tools, enabling LLMs to tackle complex\nproblems more effectively. However, there is significant potential to extend this integration to more\nadvanced symbolic reasoning systems, such as Lean 9 for mathematical theorem proving or other\ndomain-specific tools. Moreover, integratin

VectorStoreRetriever supports search types of "similarity" (default), "mmr" (maximum marginal relevance, described above), and "similarity_score_threshold". We can use the latter to threshold documents output by the retriever by similarity score.

## Create Chat Model

In [14]:
## Creat Chat Model
from langchain_ollama import ChatOllama
chat_model = ChatOllama(model="hf.co/bartowski/Meta-Llama-3.1-8B-Claude-GGUF:Q4_K_M", temperature=0.7, top_k=40)

## Create Prompt

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage

prompt = ChatPromptTemplate.from_template("""
    Given the following conversation, relevant context, and a follow up question, reply with an answer to the current question the user is asking. Return only your response to the question given the above information following the users instructions as neede
                                          
    <context>
    {context}
    </context>
                                          
    Question: {input}
    """)


## Create RAG Chain

In [ ]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# This can be replaced by calling_retriever_tool

# Create a chain to combine retrieved documents with the prompt for the LLM.
document_chain = create_stuff_documents_chain(chat_model, prompt)
# Create the final retrieval chain, which first retrieves documents and then passes them to the document_chain. 
retrieval_chain = create_retrieval_chain(retriever, document_chain)


## Test Run

In [ ]:
question = "What is Hybrid Thinking?"
response =  .invoke({"input": question})
print(response["answer"])

Based on the context provided, Hybrid Thinking appears to be an approach for improving language model performance that combines different thinking modes or techniques. The key points about Hybrid Thinking from the given information are:

- It was found to provide the best tradeoff between performance and efficiency compared to other methods like CoT (presumably Conversational Thinking).

- When applied to an Llama-3-8B-Instruct language model, it significantly outperformed the original model on all datasets tested.

- The improvement in accuracy ranged from 8% to over 23% depending on the dataset.

So in summary, Hybrid Thinking is a technique for fine-tuning or augmenting language models that appears to combine multiple thinking approaches. When applied as described in this context, it led to significant improvements in performance across various datasets compared to using just the original model or a single thinking mode.
